# Stage 3A — Formal DD Coupling Law and Q-Concentration Principle

Frozen evaluation only. This notebook reads deterministic Stage-3A outputs produced by `python -m scripts.stage3a`; it does not train, tune, or modify a waveform, and it does not use Stage 2E.

## Raw coupling law

Use the existing finite-dimensional atomic DD operator

$$B_W(\tau,\nu)=W^H\Delta(\nu)\Pi(\tau)W,\qquad b_{ij}(\tau,\nu)=[B_W(\tau,\nu)]_{ij}.$$

For $H_{raw}=\sum_p h_p\Delta(\nu_p)\Pi(\tau_p)$, $A_{ij}^{raw}=\sum_p h_p b_{ij,p}$ and

$$\mathbb E|A_{ij}^{raw}|^2=\sum_{p,q}\mathbb E[h_ph_q^*b_{ij,p}b_{ij,q}^*].$$

When gains are zero-mean, mutually uncorrelated, and independent of locations, all $p\ne q$ terms vanish. Therefore

$$C_{ij}^{raw}=\sum_p\sigma_p^2\int|w_i^H\Delta(\nu)\Pi(\tau)w_j|^2d\mu_p(\tau,\nu).$$

For iid locations and $h_p\sim\mathcal{CN}(0,1/P)$, this is exactly $C_{ij}^{raw}=Q_{ij}$. Here $Q$ is the channel-distribution-weighted second moment of the pairwise **discrete finite-dimensional DD cross-ambiguity**, not a continuous-time ambiguity function.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'outputs' / 'stage3a').exists())
OUT = ROOT / 'outputs' / 'stage3a'
diagnostics = json.loads((OUT / 'stage3a_diagnostics.json').read_text())
diagnostics['outcome'], diagnostics['interpretation']

In [ ]:
theorem = pd.read_csv(OUT / 'theorem_verification.csv')
theorem[['case', 'waveform', 'expected_q_scale', 'relative_frobenius_error', 'relative_mc_standard_error', 'within_mc_tolerance']]

## Exact project normalization and support link

The simulator uses $\alpha(H_{raw})=\sqrt N/\|H_{raw}\|_F$, hence

$$C_{ij}^{norm}=\mathbb E[\alpha^2|A_{ij}^{raw}|^2].$$

No independence between $\alpha$ and any entry is assumed. For a fixed mask $S$, normalized unitarity gives $\|A\|_F^2=N$ in every realization, so

$$\mathbb E[\mathrm{retained}(S)]=N^{-1}\sum_{(i,j)\in S}C_{ij}^{norm}$$

exactly. Replacing $C^{norm}$ by $Q$ is the empirical approximation.

In [ ]:
arrays = np.load(OUT / 'coupling_matrices.npz')
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for ax, label, key in zip(axes, ('Q', 'C raw', 'C normalized'), ('Q_Learned', 'Craw_B_P3_RAW_Learned', 'Cnorm_Learned')):
    matrix = arrays[key]
    im = ax.imshow(matrix / matrix.max(), origin='lower', cmap='magma', vmin=0, vmax=1)
    ax.set(title=label, xlabel='atom j', ylabel='atom i')
fig.colorbar(im, ax=axes, shrink=.8);


## Q concentration

For descending row entries $q_{i,(1)}\ge\cdots\ge q_{i,(N)}$,

$$r_i^Q(K)=\frac{\sum_{m=1}^Kq_{i,(m)}}{\sum_jQ_{ij}},\qquad \eta_K^Q=\frac{\sum_i\sum_{m=1}^Kq_{i,(m)}}{\sum_{ij}Q_{ij}}.$$

The row inverse-participation effective support and normalized entropy provide two additional concise diffuseness summaries.

In [ ]:
principal = pd.read_csv(OUT / 'principal_concentration_summary.csv')
curves = pd.read_csv(OUT / 'concentration_curves.csv')
display(principal)
for name, rows in curves.groupby('waveform', sort=False):
    plt.plot(rows.k, rows.eta_q, label=name)
plt.xlabel('row budget K'); plt.ylabel('eta_K^Q'); plt.xlim(1, 64); plt.ylim(0, 1.01); plt.grid(alpha=.3); plt.legend();

In [ ]:
checkpoints = pd.read_csv(OUT / 'checkpoint_concentration_summary.csv')
checkpoints[['waveform', 'training_k', 'initialization', 'validation_objective', 'q_to_c_spearman', 'eta_q', 'row_concentration_p05', 'effective_support_mean', 'q_fixed_retained_energy', 'qpsk_q_fixed_ber_18db', 'qam16_q_fixed_ber_18db']]

## Interpretation gate

1. Raw $Q=C$ is exact under the zero-mean uncorrelated-gain, gain/location-independence, iid-location, and unit-total-gain-variance assumptions above.
2. Realization normalization couples scale to entries and changes structure modestly, but does not break Q correspondence in DD-v1.
3. OFDM-start and authoritative solutions are substantially more Q-concentrated than all weak Haar checkpoints at matched K.
4. Masked $C^{norm}$ mass explains fixed-support retained energy exactly; Q concentration approximates it closely.
5. Finite-K BER tracks concentration across this small frozen set, but this is exploratory and not a BER theorem.
6. Candidate property for later work: large matched-budget $\eta_K^Q$ with a robust lower-row-tail constraint.
7. Direct optimization, sufficiency, conditioning, modulation robustness, and generalization beyond this channel/checkpoint set remain unresolved.

**Outcome A. Stop after Stage 3A; no Q loss or Stage 3B training is introduced.**